In [ ]:
#Imports
import pandas as pd
import geohash2
import numpy as np
import requests
import time
from datetime import datetime
import holidays
import os
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

Pt 1. Data Processing

In [ ]:
# Raw Data Preprocessing
raw_data = "/raw_data.csv"
df = pd.read_csv(raw_data)
df_cleaned = df.dropna(subset=['PIKCUP_LAT', 'PIKCUP_LON'])
df_cleaned = df_cleaned.drop_duplicates(subset=['PROVIDER_RIDE_ID'])
duplicate_check = df_cleaned[df_cleaned.duplicated(subset=['PROVIDER_RIDE_ID'], keep=False)]
base_data = df_cleaned.copy()
base_data = base_data[['OFFER_DATE','PIKCUP_LAT','PIKCUP_LON']]

In [ ]:
# Converting Coords to Geohashes (6 Char)
coordinate_columns = [
    ('PIKCUP_LAT', 'PIKCUP_LON')
]

def encode_geohash(lat, lon, precision=6):
    return geohash2.encode(lat, lon, precision)

for lat_col, lon_col in coordinate_columns:
    geohash_col_name = f"{lat_col.split('_')[0]}_GEOHASH"
    base_data.loc[:,geohash_col_name] = base_data.apply(lambda row:
                           encode_geohash(row[lat_col], row[lon_col], 6),
                           axis=1)

columns_to_drop = [col for pair in coordinate_columns for col in pair]
base_data.drop(columns=columns_to_drop, inplace=True)
base_data.head()

In [ ]:
# Filter Geohashes to extract data for relevant geohashes.
geohash_counts = base_data['PIKCUP_GEOHASH'].value_counts()
geohashes_to_keep = geohash_counts[geohash_counts >= 40000].index
base_data_filtered = base_data[base_data['PIKCUP_GEOHASH'].isin(geohashes_to_keep)]

Pt 2. Data Prep (adding some feautres)

In [ ]:
# Convert OFFER_DATE to datetime
base_data_filtered['OFFER_DATE'] = pd.to_datetime(base_data_filtered['OFFER_DATE'])

# Extract time-based features
base_data_filtered['time_of_day'] = base_data_filtered['OFFER_DATE'].dt.hour + base_data_filtered['OFFER_DATE'].dt.minute / 60
base_data_filtered['day_of_week'] = base_data_filtered['OFFER_DATE'].dt.dayofweek
base_data_filtered['month'] = base_data_filtered['OFFER_DATE'].dt.month

# Normalize and apply sine and cosine transformations
# base_data_filtered['time_of_day_sin'] = np.sin(2 * np.pi * base_data_filtered['time_of_day'] / 24)
# base_data_filtered['time_of_day_cos'] = np.cos(2 * np.pi * base_data_filtered['time_of_day'] / 24)

# base_data_filtered['day_of_week_sin'] = np.sin(2 * np.pi * base_data_filtered['day_of_week'] / 7)
# base_data_filtered['day_of_week_cos'] = np.cos(2 * np.pi * base_data_filtered['day_of_week'] / 7)

# base_data_filtered['month_sin'] = np.sin(2 * np.pi * base_data_filtered['month'] / 12)
# base_data_filtered['month_cos'] = np.cos(2 * np.pi * base_data_filtered['month'] / 12)

In [ ]:
# Adding Weather

# Function to fetch weather data from Open-Meteo API
def fetch_weather_data(lat, lon, date):
    url = f"https://api.open-meteo.com/v1/forecast"
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date,
        'end_date': date,
        'hourly': 'temperature_2m,precipitation'
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        return None

# New York Coordinates
latitude = 40.7128
longitude = -74.0060

base_data_filtered['OFFER_DATE'] = pd.to_datetime(base_data_filtered['OFFER_DATE'])


weather_cache = {}
temperatures = []
precipitations = []

for index, row in base_data_filtered.iterrows():
    date = row['OFFER_DATE'].date()
    if date not in weather_cache:
        weather_data = fetch_weather_data(latitude, longitude, date)
        if weather_data:
            hourly_data = weather_data['hourly']
            # Store hourly weather data
            weather_cache[date] = {
                'temperature': hourly_data['temperature_2m'],
                'precipitation': hourly_data['precipitation']
            }
        else:
            weather_cache[date] = {
                'temperature': [None] * 24,
                'precipitation': [None] * 24
            }
        time.sleep(1)

    hour = row['OFFER_DATE'].hour
    temperatures.append(weather_cache[date]['temperature'][hour])
    precipitations.append(weather_cache[date]['precipitation'][hour])

base_data_filtered['temperature'] = temperatures
base_data_filtered['precipitation'] = precipitations

# Manually Adding missing weather values

weather_data = pd.read_csv('/weather.csv')

weather_data['time'] = pd.to_datetime(weather_data['time'])
weather_data['date'] = weather_data['time'].dt.date
weather_data['hour'] = weather_data['time'].dt.hour

weather_data.rename(columns={'temperature_2m (°C)': 'temperature', 'precipitation (mm)': 'precipitation'}, inplace=True)


base_data_filtered['date'] = base_data_filtered['OFFER_DATE'].dt.date
base_data_filtered['hour'] = base_data_filtered['OFFER_DATE'].dt.hour

weather_dict = weather_data.set_index(['date', 'hour']).to_dict(orient='index')

# Loop through base_data_filtered to update missing entries
for i, row in base_data_filtered.iterrows():
    if pd.isna(row['temperature']) or pd.isna(row['precipitation']):
        date = row['date']
        hour = row['hour']
        if (date, hour) in weather_dict:
            if pd.isna(row['temperature']):
                base_data_filtered.at[i, 'temperature'] = weather_dict[(date, hour)]['temperature']
            if pd.isna(row['precipitation']):
                base_data_filtered.at[i, 'precipitation'] = weather_dict[(date, hour)]['precipitation']

base_data_filtered.drop(columns=['date', 'hour'], inplace=True)

In [ ]:
# Split Data based on geohashes

geohash_dfs = {geohash: base_data_filtered[base_data_filtered['PIKCUP_GEOHASH'] == geohash] for geohash in base_data_filtered['PIKCUP_GEOHASH'].unique()}

# Display the first few rows of each DataFrame
for geohash, df in geohash_dfs.items():
    print(f"Geohash: {geohash}")
    print(df.head())
    print("\n")

# Save Data for each geohash
for geohash, df in geohash_dfs.items():
    df.to_csv(f'/Data2/{geohash}.csv', index=False)

Model Training

In [ ]:
data2 = '/Data2/'

In [ ]:
# Reload the saved data, can be skipped if the entire code block is run
geohash_dfs = {}

for filename in os.listdir(data2):
    if filename.endswith(".csv"):
        geohash = filename.split('.')[0]

        df = pd.read_csv(os.path.join(data2, filename))

        geohash_dfs[geohash] = df

# Verify the loaded DataFrames
for geohash, df in geohash_dfs.items():
    print(f"Geohash: {geohash}")
    print(df.head())
    print("\n")

In [ ]:
# Final Preprocessing function

def preprocess_data(df):
    trialdf = df[['OFFER_DATE', 'time_of_day', 'day_of_week', 'month', 'temperature', 'precipitation']]

    trialdf = trialdf.sort_values(by='OFFER_DATE')

    trialdf['OFFER_DATE'] = pd.to_datetime(trialdf['OFFER_DATE'])

    trialdf['next_OFFER_DATE'] = trialdf['OFFER_DATE'].shift(-1)

    trialdf['time_to_next_ride'] = (trialdf['next_OFFER_DATE'] - trialdf['OFFER_DATE']).dt.total_seconds()

    trialdf.drop(columns=['next_OFFER_DATE'], inplace=True)

    trialdf.dropna(subset=['time_to_next_ride'], inplace=True)

    us_holidays = holidays.US()
    trialdf['is_holiday'] = trialdf['OFFER_DATE'].dt.date.apply(lambda x: 1 if x in us_holidays else 0)

    filtered_df = trialdf.copy()

    return filtered_df

# Model Training Function
def train_model(df, save_path):
    # Select the features and target
    features = df[['time_of_day', 'day_of_week', 'month', 'temperature', 'precipitation', 'is_holiday']]
    target = df['time_to_next_ride']

    # Normalize the features
    feature_scaler = MinMaxScaler()
    features_scaled = feature_scaler.fit_transform(features)

    # Normalize the target
    target_scaler = MinMaxScaler()
    target_scaled = target_scaler.fit_transform(target.values.reshape(-1, 1))

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(features_scaled, target_scaled, test_size=0.2, random_state=42)

    # Build the model
    model = Sequential()
    model.add(Dense(units=128, activation='relu', input_shape=(features_scaled.shape[1],)))
    model.add(Dense(units=32, activation='relu'))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))

    # Compile the model
    model.compile(optimizer='adam', loss='mse')

    # Train the model
    model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

    # Save the model
    model.save(save_path)

    # Evaluate the model
    loss = model.evaluate(X_test, y_test, verbose=1)
    print(f'Test Loss: {loss}')

    return model, feature_scaler, target_scaler

In [ ]:
# Loop for training and saving models

model_dir = '/models/'

# Iterate through the dictionary of DataFrames
for geohash, df in geohash_dfs.items():
    print(f"Processing Geohash: {geohash}")

    # Preprocess the data
    preprocessed_df = preprocess_data(df)

    # Train the model and save it
    model_save_path = os.path.join(model_dir, f"{geohash}_model.keras")
    model, feature_scaler, target_scaler = train_model(preprocessed_df, model_save_path)

    print(f"Model for Geohash {geohash} saved successfully.")